In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# OLD8 双状态静态／换序局部载体 V1

发布并绑定不可变 source SHA 后直接 Run all。固定同一开发 source/seed2026092501、同模型 revision、shared initial/pristine scheduler；六臂 OFF、PAYLOAD_MULTI、STATE_A、STATE_B、STATE_AB、STATE_BA 各真实50步，控制固定 MULTI25..49。

A/B 只取原 OLD8 key 码本排序第0/1个32-chip字，不看保存响应。换序边界固定在起点23：1..22与23..45交换；u23..25保留真实四-age混合过渡。OLD8四个8x8支持、真实余弦基、channel4、四age、alpha、payload及 eta696/cap1 shrink-only 更新均不变。每臂记录实际 raw/applied norm；相同 cap 不称等实际预算，不匹配、不放大。

每个收到的保存观测与 key 在不知道 arm、真序列、消息、writer latent/sidecar、同源OFF或攻击真值时，同时评分 AA/BB/AB/BA。ABS 是主四序列整体可辨识诊断；ADJACENT_DIFFERENCE 是预先保留的独立诊断，恒A/恒B在部分支持上的精确等价完整保留，不要求 DIFF 也唯一。逐窗四模板软代价及每个 window×age 的 A/B 32-chip cost margin 全部保存；首端、转换区、tie、缺支持、nonfinite和失败不删除。盲 readout 先保存封存，随后才加载该文件做 truth/message join。配对差仅事后质量诊断。

DIRECT_RGB8、RAW420、MP4 均从各臂同一保存 RGB8 raster进入，generation与VAE/media分子进程释放。固定g0/R44是公开诊断相位，不能称未知起点或任意编辑盲同步。重复payload不是时间相关payload。所有 acceptance=false；EXECUTION_COMPLETE只表示固定工程执行完整，不是科学PASS或已成功盲同步。

Published source: `af1a5051a1ecfaa347d748068697eabd54c75031`.


In [ ]:
SOURCE_SHA = 'af1a5051a1ecfaa347d748068697eabd54c75031'
FIXED = {'source_cases': 1, 'arms': 6, 'channels': 3, 'generated': 6, 'generation_steps': 300, 'controlled_steps': 125, 'writer_sidecars': 150, 'rasters': 6, 'transport': 18, 'underlying_observations': 18, 'normalized': 18, 'keys': 2, 'keyed_projections': 36, 'sequence_score_records': 72, 'sequence_candidate_scores': 288, 'absolute_local_candidate_costs': 6336, 'difference_local_candidate_costs': 6192, 'component_state_costs': 12240, 'sequence_posthoc': 72, 'local_posthoc': 36, 'payload_reads': 36, 'payload_evaluations': 72, 'quality_pairs': 45}
from pathlib import Path
import datetime,hashlib,json,os,subprocess,sys,traceback
if SOURCE_SHA is None:raise RuntimeError('UNPUBLISHED_DRAFT: publish source and rebuild with its immutable SHA before Run all')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Video-Local-Fourier-RM-OLD8-Two-State-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-LOCAL-FOURIER-RM-OLD8-TWO-STATE-'+STAMP)
PYTHON=sys.executable
ARMS=('OFF','PAYLOAD_MULTI','STATE_A_MULTI','STATE_B_MULTI','STATE_AB_MULTI','STATE_BA_MULTI')
CHANNELS=('DIRECT_RGB8','RAW420','MP4');KEYS=('K0','K1');MODES=('ABSOLUTE_CONTROL','ADJACENT_DIFFERENCE')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED,
 generation={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 writer_diagnostics={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ARMS for i in range(25,50)},
 terminal_inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},rasters={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 transport={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},
 normalized={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},
 projections={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 sequence_reads={a+'/'+c+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_sequence_accepted':False} for a in ARMS for c in CHANNELS for k in KEYS for m in MODES},
 payload_reads={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 sequence_posthoc={a+'/'+c+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS for m in MODES},
 local_posthoc={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 payload_evaluations={a+'/'+c+'/'+k+'/'+t:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS for t in ('REGISTERED','WRONG_MESSAGE')},
 quality={c+'/'+a+'_vs_'+b:{'status':'NOT_RUN_SETUP'} for c in CHANNELS for i,a in enumerate(ARMS) for b in ARMS[:i]})
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Fixed output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind published source SHA before Run all');failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    import shutil
    tools={name:shutil.which(name) for name in ('ffmpeg','ffprobe')}
    tool_codes={name:logged([path,'-version'],name.upper()+'_PROBE',check=False) if path else None for name,path in tools.items()}
    if any(code!=0 for code in tool_codes.values()):
        try:
            logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
            logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
        except Exception as repair_error:print('Media-tool repair error retained:',repair_error)
    setup['media_tool_probe_returncodes']=tool_codes
    torch_ok=logged([PYTHON,'-c','import torch; print(torch.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:logged([PYTHON,'-m','pip','install','--upgrade','torch'],'TORCH_REPAIR',check=False)
    vae_probe='from diffusers import AutoencoderKLWan,WanPipeline; print(AutoencoderKLWan.__name__,WanPipeline.__name__)'
    vae_ok=logged([PYTHON,'-c',vae_probe],'VAE_IMPORT_PROBE',check=False)==0
    if not vae_ok:
        logged([PYTHON,'-m','pip','install','diffusers>=0.35.1,<0.40','transformers>=4.45,<5','accelerate>=0.34','safetensors>=0.4.5'],'VAE_DEPENDENCY_REPAIR',check=False)
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual VAE import/load determines availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    environment_code='import json,torch; from experiments.wan_state_clock.video_local_fourier_rm_old8_two_state_v1_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',environment_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_separate_generation_media_workers')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_local_fourier_rm_old8_two_state_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; setup denominator retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Calls:',result['calls'],'Workers:',result['workers'],'Blind SHA:',result.get('blind_receiver_sha256'))
print('A/B selection:',result['selection'],'Shared before25:',result.get('before_step25_identity'))
for arm,row in result['generation'].items():print('Writer:',arm,row['status'],(row.get('receipt') or {}).get('control_norm_statistics'))
for name,row in result['sequence_posthoc'].items():print('Sequence:',name,row['status'],'rank=',row.get('true_rank'),'delta=',row.get('true_delta'),'top=',row.get('top'))
for name,row in result['local_posthoc'].items():print('Local A/B:',name,row['status'],'scored=',row.get('scored'),'truth-in-top=',row.get('truth_in_top'),'unique=',row.get('unique_truth'))
for name,row in result['payload_evaluations'].items():print('Repeated payload:',name,row['status'],'bit errors=',row.get('bit_errors'))
print('Saved-RGB quality diagnostics:',result['quality'])
from IPython.display import Video,display
for arm in ARMS:
    video=result['transport'].get(arm+'/MP4',{}).get('events',{}).get('mp4',{});path=video.get('path')
    if video.get('status')=='SAVED' and isinstance(path,str) and Path(path).is_file():
        try:print('Saved MP4 preview:',arm);display(Video(str(path),embed=True))
        except Exception as preview_error:print('Preview unavailable; result retained:',arm,preview_error)
print('Failures:',result['failures']);print('Evidence ceiling:',result['evidence_ceiling']);print('Retained result:',RESULT_PATH)
